In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

# a base carrega por URL, sem upload e sem credencial
URL = ("https://raw.githubusercontent.com/mwaskom/seaborn-data/"
       "master/mpg.csv")
df = pd.read_csv(URL)          # (398, 9)
d  = df.dropna(subset=["horsepower"]).copy()   # (392, 9)

In [ ]:
modelo = smf.ols("mpg ~ weight", data=df).fit()

beta0 = float(modelo.params["Intercept"])
beta1 = float(modelo.params["weight"])
ic_beta1 = modelo.conf_int().loc["weight"]
r2 = float(modelo.rsquared)
erro_residual = float(np.sqrt(modelo.scale))
media_mpg = float(df["mpg"].mean())
estatistica_f = float(modelo.fvalue)
t_weight = float(modelo.tvalues["weight"])

variacao_1000_lb = beta1 * 1000

modelo_d = smf.ols("mpg ~ weight", data=d).fit()

print(modelo.summary())

print("\nQuestao 1 - respostas")
print(
       f"a) Equacao estimada: mpg = {beta0:.4f} {beta1:+.6f} * weight"
)
print(
       f"   Interpretacao de beta1: um aumento de 1.000 libras no peso esta associado a {variacao_1000_lb:.2f} mpg a menos, em media."
)
print(
       f"b) Intercepto = {beta0:.4f}. Ele nao representa o consumo de um carro com peso zero, porque peso zero esta fora do dominio observado e nao faz sentido fisico para um carro."
)
print(
       "   Na pratica, o intercepto e apenas a constante matematica que ancora a reta de regressao no eixo vertical."
)
print(
       f"c) IC de 95% para beta1: [{float(ic_beta1.iloc[0]):.6f}, {float(ic_beta1.iloc[1]):.6f}] mpg por libra."
)
print(
       f"   Em linguagem de negocio: cada libra adicional de peso esta associada a uma queda media entre {float(ic_beta1.iloc[1]) * 1000:.2f} e {float(ic_beta1.iloc[0]) * 1000:.2f} mpg por 1.000 libras."
)
print(
       f"d) R^2 = {r2:.4f}; erro-padrao residual = {erro_residual:.4f} mpg; media de mpg = {media_mpg:.4f}."
)
print(
       "   Para uma reuniao de negocio, eu levaria o erro-padrao residual, porque ele esta na mesma unidade do consumo e mostra o erro tipico de previsao em mpg."
)
print(
       f"e) F = {estatistica_f:.4f}; t(weight) = {t_weight:.4f}; t(weight)^2 = {t_weight ** 2:.4f}."
)
print(
       "   Neste modelo com um unico regressor, a estatistica F do teste global e exatamente o quadrado da estatistica t do coeficiente angular."
)
print(
       f"\nObs.: usando d (apos remover missing em horsepower), os valores da imagem reaparecem: intercepto = {float(modelo_d.params['Intercept']):.4f}, F = {float(modelo_d.fvalue):.2f}, t(weight) = {float(modelo_d.tvalues['weight']):.4f}."
)

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.692
Model:                            OLS   Adj. R-squared:                  0.691
Method:                 Least Squares   F-statistic:                     888.9
Date:                Sun, 20 Sep 2026   Prob (F-statistic):          2.97e-103
Time:                        17:05:33   Log-Likelihood:                -1148.4
No. Observations:                 398   AIC:                             2301.
Df Residuals:                     396   BIC:                             2309.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     46.3174      0.795     58.243      0.0

In [ ]:
modelo_weight = smf.ols("mpg ~ weight", data=d).fit()
modelo_horsepower = smf.ols("mpg ~ horsepower", data=d).fit()
modelo_displacement = smf.ols("mpg ~ displacement", data=d).fit()
modelo_triplo = smf.ols("mpg ~ weight + horsepower + displacement", data=d).fit()

tabela_q2 = pd.DataFrame(
       [
              {
                     "preditor": "weight",
                     "situacao": "sozinho",
                     "coef": float(modelo_weight.params["weight"]),
                     "erro_padrao": float(modelo_weight.bse["weight"]),
                     "t": float(modelo_weight.tvalues["weight"]),
                     "p_valor": float(modelo_weight.pvalues["weight"]),
              },
              {
                     "preditor": "horsepower",
                     "situacao": "sozinho",
                     "coef": float(modelo_horsepower.params["horsepower"]),
                     "erro_padrao": float(modelo_horsepower.bse["horsepower"]),
                     "t": float(modelo_horsepower.tvalues["horsepower"]),
                     "p_valor": float(modelo_horsepower.pvalues["horsepower"]),
              },
              {
                     "preditor": "displacement",
                     "situacao": "sozinho",
                     "coef": float(modelo_displacement.params["displacement"]),
                     "erro_padrao": float(modelo_displacement.bse["displacement"]),
                     "t": float(modelo_displacement.tvalues["displacement"]),
                     "p_valor": float(modelo_displacement.pvalues["displacement"]),
              },
              {
                     "preditor": "weight",
                     "situacao": "acompanhado",
                     "coef": float(modelo_triplo.params["weight"]),
                     "erro_padrao": float(modelo_triplo.bse["weight"]),
                     "t": float(modelo_triplo.tvalues["weight"]),
                     "p_valor": float(modelo_triplo.pvalues["weight"]),
              },
              {
                     "preditor": "horsepower",
                     "situacao": "acompanhado",
                     "coef": float(modelo_triplo.params["horsepower"]),
                     "erro_padrao": float(modelo_triplo.bse["horsepower"]),
                     "t": float(modelo_triplo.tvalues["horsepower"]),
                     "p_valor": float(modelo_triplo.pvalues["horsepower"]),
              },
              {
                     "preditor": "displacement",
                     "situacao": "acompanhado",
                     "coef": float(modelo_triplo.params["displacement"]),
                     "erro_padrao": float(modelo_triplo.bse["displacement"]),
                     "t": float(modelo_triplo.tvalues["displacement"]),
                     "p_valor": float(modelo_triplo.pvalues["displacement"]),
              },
       ]
)

correlacoes = d[["weight", "horsepower", "displacement"]].corr()

print("\nQuestao 2 - respostas")
print("a) Tabela comparando cada preditor sozinho versus acompanhado:")
print(tabela_q2.round(6).to_string(index=False))
print(
       "b) displacement muda mais drasticamente: sozinho ele tem relacao negativa forte com mpg, mas na regressao multipla seu coeficiente fica perto de zero e deixa de ser estatisticamente significativo."
)
print(
       "   horsepower tambem perde magnitude quando weight entra no modelo, mas continua significativo; portanto, ele ainda agrega informacao propria depois do controle pelos demais preditores."
)
print("c) Matriz de correlacao entre os tres preditores:")
print(correlacoes.round(4).to_string())
print(
       "   Os erros-padrao aumentam porque os tres preditores sao muito correlacionados entre si, especialmente weight com displacement e displacement com horsepower."
)
print(
       "   Essa multicolinearidade dificulta separar o efeito parcial de cada variavel e infla a incerteza dos coeficientes na regressao multipla."
)
print(
       f"d) R^2 sobe de {float(modelo_weight.rsquared):.4f} para {float(modelo_triplo.rsquared):.4f}, mas o R^2 ajustado sobe pouco, de {float(modelo_weight.rsquared_adj):.4f} para {float(modelo_triplo.rsquared_adj):.4f}."
)
print(
       f"   O erro-padrao residual cai apenas de {float(np.sqrt(modelo_weight.scale)):.4f} para {float(np.sqrt(modelo_triplo.scale)):.4f} mpg."
)
print(
       "   Entao, o ganho de ajuste existe, mas e modesto; manter os tres so vale a pena se houver interesse analitico em controle simultaneo, porque displacement praticamente nao contribui depois que weight e horsepower entram."
)
print(
       "e) A conclusao esta errada porque um p-valor alto para displacement na regressao multipla nao prova ausencia de relacao com consumo."
)
print(
       "   O teste apenas indica que, depois de controlar por weight e horsepower, nao ha evidencia suficiente de efeito parcial adicional de displacement nesta amostra."
)
print(
       "   Como essas variaveis sao fortemente correlacionadas, o efeito de displacement pode estar compartilhado com as outras duas, o que reduz a precisao do teste individual."
)


Questao 2 - respostas
a) Tabela comparando cada preditor sozinho versus acompanhado:
    preditor    situacao      coef  erro_padrao          t  p_valor
      weight     sozinho -0.007647     0.000258 -29.645082 0.000000
  horsepower     sozinho -0.157845     0.006446 -24.489135 0.000000
displacement     sozinho -0.060051     0.002240 -26.808153 0.000000
      weight acompanhado -0.005352     0.000712  -7.512547 0.000000
  horsepower acompanhado -0.041674     0.012814  -3.252270 0.001245
displacement acompanhado -0.005769     0.006582  -0.876468 0.381318
b) displacement muda mais drasticamente: sozinho ele tem relacao negativa forte com mpg, mas na regressao multipla seu coeficiente fica perto de zero e deixa de ser estatisticamente significativo.
   horsepower tambem perde magnitude quando weight entra no modelo, mas continua significativo; portanto, ele ainda agrega informacao propria depois do controle pelos demais preditores.
c) Matriz de correlacao entre os tres preditores:
     

In [ ]:
modelo_origem = smf.ols("mpg ~ C(origin)", data=d).fit()
modelo_peso_origem = smf.ols("mpg ~ weight + C(origin)", data=d).fit()
teste_f_origem = modelo_peso_origem.f_test(
       "C(origin)[T.japan] = 0, C(origin)[T.usa] = 0"
)

medias_mpg_origem = d.groupby("origin")["mpg"].agg(["count", "mean"])
medias_peso_origem = d.groupby("origin")["weight"].mean()

intercepto_origem = float(modelo_origem.params["Intercept"])
coef_japan = float(modelo_origem.params["C(origin)[T.japan]"])
coef_usa = float(modelo_origem.params["C(origin)[T.usa]"])

print("\nQuestao 3 - respostas")
print(
       "a) Em mpg ~ C(origin), o statsmodels escolhe europe como categoria de referencia."
)
print(
       "   O criterio e a codificacao por tratamento: uma categoria vira base e as demais entram como diferencas em relacao a ela; aqui, europe aparece como base na parametrizacao retornada."
)
print(
       f"   Intercepto = {intercepto_origem:.4f}, que e a media de mpg dos carros europeus."
)
print(
       f"   Coeficiente de japan = {coef_japan:.4f}, ou seja, carros do japao tem em media {coef_japan:.4f} mpg a mais do que carros europeus."
)
print(
       f"   Coeficiente de usa = {coef_usa:.4f}, ou seja, carros dos EUA tem em media {abs(coef_usa):.4f} mpg a menos do que carros europeus."
)
print("b) Medias de mpg por grupo:")
print(medias_mpg_origem.round(4).to_string())
print(
       f"   Conta para europe: {intercepto_origem:.4f}."
)
print(
       f"   Conta para japan: {intercepto_origem:.4f} + {coef_japan:.4f} = {intercepto_origem + coef_japan:.4f}."
)
print(
       f"   Conta para usa: {intercepto_origem:.4f} {coef_usa:+.4f} = {intercepto_origem + coef_usa:.4f}."
)
print(
       "   Essas contas reproduzem exatamente as medias observadas de mpg em cada procedencia."
)
print(
       f"c) Em mpg ~ weight + C(origin), os p-valores individuais das dummies sao {float(modelo_peso_origem.pvalues['C(origin)[T.japan]']):.4f} para japan e {float(modelo_peso_origem.pvalues['C(origin)[T.usa]']):.4f} para usa, ambos acima de 5%."
)
print(
       f"   Mas o teste F do bloco das duas dummies da F = {float(teste_f_origem.fvalue):.4f} com p-valor = {float(teste_f_origem.pvalue):.4f}."
)
print(
       "   Portanto, as dummies nao sao fortes isoladamente, mas sao relevantes em conjunto."
)
print(
       "   Isso acontece porque cada dummy separada disputa explicacao com weight e com a outra dummy, enquanto o teste F verifica se o bloco inteiro de origem adiciona informacao ao modelo."
)
print("d) Peso medio por procedencia:")
print(medias_peso_origem.round(4).to_string())
print(
       "   O efeito de origin encolhe muito ao controlar por weight porque boa parte da diferenca de consumo entre procedencias passa pelo fato de os carros dos EUA serem, em media, muito mais pesados, e os japoneses e europeus serem mais leves."
)
print(
       "   Quando weight entra no modelo, ele absorve grande parte dessa diferenca estrutural, e sobra menos efeito exclusivo para as dummies de origem."
)



Questao 3 - respostas
a) Em mpg ~ C(origin), o statsmodels escolhe europe como categoria de referencia.
   O criterio e a codificacao por tratamento: uma categoria vira base e as demais entram como diferencas em relacao a ela; aqui, europe aparece como base na parametrizacao retornada.
   Intercepto = 27.6029, que e a media de mpg dos carros europeus.
   Coeficiente de japan = 2.8477, ou seja, carros do japao tem em media 2.8477 mpg a mais do que carros europeus.
   Coeficiente de usa = -7.5695, ou seja, carros dos EUA tem em media 7.5695 mpg a menos do que carros europeus.
b) Medias de mpg por grupo:
        count     mean
origin                
europe     68  27.6029
japan      79  30.4506
usa       245  20.0335
   Conta para europe: 27.6029.
   Conta para japan: 27.6029 + 2.8477 = 30.4506.
   Conta para usa: 27.6029 -7.5695 = 20.0335.
   Essas contas reproduzem exatamente as medias observadas de mpg em cada procedencia.
c) Em mpg ~ weight + C(origin), os p-valores individuais das d

In [ ]:
modelo_completo = smf.ols(
       "mpg ~ weight + horsepower + displacement + acceleration + model_year + C(origin)",
       data=d,
).fit()
modelo_explicativo = smf.ols("mpg ~ weight + model_year + C(origin)", data=d).fit()

X_completo = pd.get_dummies(
       d[["weight", "horsepower", "displacement", "acceleration", "model_year", "origin"]],
       columns=["origin"],
       drop_first=True,
)
X_explicativo = pd.get_dummies(
       d[["weight", "model_year", "origin"]],
       columns=["origin"],
       drop_first=True,
)
y = d["mpg"]

cv_10 = KFold(n_splits=10, shuffle=True, random_state=42)
rmse_cv_completo = float(
       (-cross_val_score(
              LinearRegression(),
              X_completo,
              y,
              cv=cv_10,
              scoring="neg_root_mean_squared_error",
       )).mean()
)
rmse_cv_explicativo = float(
       (-cross_val_score(
              LinearRegression(),
              X_explicativo,
              y,
              cv=cv_10,
              scoring="neg_root_mean_squared_error",
       )).mean()
)

comparacoes_cv = []
for semente in range(30):
       cv = KFold(n_splits=10, shuffle=True, random_state=semente)
       rmse_full = (-cross_val_score(
              LinearRegression(),
              X_completo,
              y,
              cv=cv,
              scoring="neg_root_mean_squared_error",
       )).mean()
       rmse_explain = (-cross_val_score(
              LinearRegression(),
              X_explicativo,
              y,
              cv=cv,
              scoring="neg_root_mean_squared_error",
       )).mean()
       comparacoes_cv.append((rmse_full, rmse_explain))

comparacoes_cv = np.array(comparacoes_cv)

print("\nQuestao 4 - respostas")
print(
       f"a) No modelo completo, o coeficiente de displacement e positivo: {float(modelo_completo.params['displacement']):.6f}."
)
print(
       "   Esse sinal nao e plausivel do ponto de vista da engenharia se a leitura for causal, porque motores maiores nao deveriam melhorar consumo mantendo o resto fixo por um mecanismo fisico simples."
)
print(
       "   O resultado e produzido principalmente por multicolinearidade: displacement, weight e horsepower carregam informacao muito parecida, entao o modelo tenta separar efeitos parciais quase redundantes e pode inverter sinais."
)
print(
       "b) Isso nao e contradicao porque previsao e explicacao nao exigem a mesma coisa."
)
print(
       "   Um modelo pode prever bem se combinar variaveis que, em conjunto, aproximam bem mpg, mesmo que alguns coeficientes individuais fiquem instaveis ou pouco interpretaveis."
)
print(
       f"   Aqui, o modelo completo tem bom ajuste global (R^2 = {float(modelo_completo.rsquared):.4f}; erro residual = {float(np.sqrt(modelo_completo.scale)):.4f}), apesar do coeficiente implausivel de displacement."
)
print(
       "c) Para o Pedido A, eu entregaria mpg ~ weight + model_year + C(origin)."
)
print(
       "   Criterio: para decidir investimento em engenharia, o coeficiente de weight precisa ser interpretavel e estavel; por isso, excluo proxies mecanicamente coladas a weight, como displacement e horsepower, que distorcem a leitura do efeito parcial."
)
print(
       "   Para o Pedido B, eu entregaria o modelo completo mpg ~ weight + horsepower + displacement + acceleration + model_year + C(origin)."
)
print(
       "   Criterio: quando o objetivo e prever os 500 carros, vale usar o conjunto mais rico de preditores disponiveis e avaliar o desempenho por erro de previsao, nao pela plausibilidade isolada dos coeficientes."
)
print(
       f"   Na validacao cruzada 10-fold, os dois modelos ficam muito proximos: RMSE do completo = {rmse_cv_completo:.4f} e RMSE do explicativo = {rmse_cv_explicativo:.4f}."
)
print(
       f"   Em 30 reamostragens, o RMSE medio ficou {float(comparacoes_cv[:, 0].mean()):.4f} para o completo e {float(comparacoes_cv[:, 1].mean()):.4f} para o explicativo, mostrando que a diferenca preditiva pratica e pequena."
)
print(
       "d) No Pedido A, eu acrescentaria interpretacao economica e tecnica dos coeficientes, incerteza dos efeitos e advertencias sobre quais controles permitem ou atrapalham a leitura causal do peso."
)
print(
       "   No Pedido B, eu acrescentaria metricas de validacao preditiva, faixa esperada de erro nas previsoes e requisitos operacionais de entrada para pontuar novos carros; em compensacao, a interpretacao causal fina dos coeficientes passa a ser secundaria."
)


Questao 4 - respostas
a) No modelo completo, o coeficiente de displacement e positivo: 0.016436.
   Esse sinal nao e plausivel do ponto de vista da engenharia se a leitura for causal, porque motores maiores nao deveriam melhorar consumo mantendo o resto fixo por um mecanismo fisico simples.
   O resultado e produzido principalmente por multicolinearidade: displacement, weight e horsepower carregam informacao muito parecida, entao o modelo tenta separar efeitos parciais quase redundantes e pode inverter sinais.
b) Isso nao e contradicao porque previsao e explicacao nao exigem a mesma coisa.
   Um modelo pode prever bem se combinar variaveis que, em conjunto, aproximam bem mpg, mesmo que alguns coeficientes individuais fiquem instaveis ou pouco interpretaveis.
   Aqui, o modelo completo tem bom ajuste global (R^2 = 0.8231; erro residual = 3.3122), apesar do coeficiente implausivel de displacement.
c) Para o Pedido A, eu entregaria mpg ~ weight + model_year + C(origin).
   Criterio: para

In [ ]:
dados_holdout = pd.get_dummies(
       d[
              [
                     "mpg",
                     "weight",
                     "horsepower",
                     "displacement",
                     "acceleration",
                     "model_year",
                     "origin",
                     "name",
              ]
       ],
       columns=["origin"],
       drop_first=True,
)

X_holdout = dados_holdout.drop(columns=["mpg", "name"])
y_holdout = dados_holdout["mpg"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
       X_holdout,
       y_holdout,
       test_size=0.25,
       random_state=42,
)

modelo_holdout = LinearRegression().fit(X_treino, y_treino)
pred_teste = modelo_holdout.predict(X_teste)

mae_teste = mean_absolute_error(y_teste, pred_teste)
rmse_teste = mean_squared_error(y_teste, pred_teste) ** 0.5
r2_teste = r2_score(y_teste, pred_teste)

baseline_teste = np.repeat(y_treino.mean(), len(y_teste))
mae_baseline = mean_absolute_error(y_teste, baseline_teste)
rmse_baseline = mean_squared_error(y_teste, baseline_teste) ** 0.5
r2_baseline = r2_score(y_teste, baseline_teste)

reducao_mae = (1 - mae_teste / mae_baseline) * 100
reducao_rmse = (1 - rmse_teste / rmse_baseline) * 100

X_weight = d[["weight"]]
y_weight = d["mpg"]

X_weight_treino, X_weight_teste, y_weight_treino, y_weight_teste = train_test_split(
       X_weight,
       y_weight,
       test_size=0.25,
       random_state=42,
)

modelo_holdout_weight = LinearRegression().fit(X_weight_treino, y_weight_treino)
pred_teste_weight = modelo_holdout_weight.predict(X_weight_teste)

mae_teste_weight = mean_absolute_error(y_weight_teste, pred_teste_weight)
rmse_teste_weight = mean_squared_error(y_weight_teste, pred_teste_weight) ** 0.5
r2_teste_weight = r2_score(y_weight_teste, pred_teste_weight)

residuos_teste = y_teste.to_numpy() - pred_teste
erros_absolutos = np.abs(residuos_teste)
pos_maior_erro = int(np.argmax(erros_absolutos))
indice_maior_erro = y_teste.index[pos_maior_erro]
carro_maior_erro = d.loc[
       indice_maior_erro,
       ["name", "mpg", "weight", "horsepower", "displacement", "acceleration", "model_year", "origin"],
]

plt.figure(figsize=(8, 5))
plt.scatter(pred_teste, residuos_teste, alpha=0.75, edgecolors="none")
plt.axhline(0, color="firebrick", linestyle="--", linewidth=1.2)
plt.xlabel("Valor ajustado no teste")
plt.ylabel("Residuo")
plt.title("Questao 5 - Residuo x valor ajustado (teste)")
plt.tight_layout()
plt.savefig("q5_residuo_vs_ajustado.png", dpi=150)
plt.close()

modelo_simples_q5 = smf.ols("mpg ~ weight", data=d).fit()
quintis_ajustado = pd.qcut(modelo_simples_q5.fittedvalues, 5, duplicates="drop")
media_residuo_por_quintil = modelo_simples_q5.resid.groupby(quintis_ajustado, observed=False).mean()

print("\nQuestao 5 - respostas")
print(
       "2) Foi usada a separacao treino/teste com train_test_split(test_size=0.25, random_state=42) no modelo completo com as cinco variaveis numericas e dummies de origin com drop_first=True."
)
print(
       f"3) No conjunto de teste, o modelo completo obteve MAE = {mae_teste:.4f}, RMSE = {rmse_teste:.4f} e R^2 = {r2_teste:.4f}."
)
print(
       f"4) O baseline que sempre prevê a media do treino obteve, no teste, MAE = {mae_baseline:.4f}, RMSE = {rmse_baseline:.4f} e R^2 = {r2_baseline:.4f}."
)
print(
       f"   Em relacao ao baseline, o modelo reduziu o MAE em {reducao_mae:.2f}% e o RMSE em {reducao_rmse:.2f}%."
)
print(
       f"5) Repetindo o procedimento com apenas weight, no teste temos MAE = {mae_teste_weight:.4f}, RMSE = {rmse_teste_weight:.4f} e R^2 = {r2_teste_weight:.4f}."
)
print(
       f"   Assim, as outras variaveis acrescentam ganho de {mae_teste_weight - mae_teste:.4f} pontos em MAE, {rmse_teste_weight - rmse_teste:.4f} em RMSE e {r2_teste - r2_teste_weight:.4f} em R^2 sobre o modelo so com weight."
)
print(
       "6) O grafico de residuo x valor ajustado foi salvo em q5_residuo_vs_ajustado.png."
)
print(
       f"   O carro de maior erro absoluto no teste foi {carro_maior_erro['name']}, com mpg observado = {float(carro_maior_erro['mpg']):.1f}, previsao = {float(pred_teste[pos_maior_erro]):.4f} e erro absoluto = {float(erros_absolutos[pos_maior_erro]):.4f}."
)
print(
       f"   Ele chama atencao por ser um {carro_maior_erro['origin']} de {int(carro_maior_erro['model_year'])}, muito leve ({float(carro_maior_erro['weight']):.0f}), com baixa cilindrada ({float(carro_maior_erro['displacement']):.1f}) e baixa potencia ({float(carro_maior_erro['horsepower']):.1f}), mas consumo observado bem pior do que o previsto."
)
print("7) Media do residuo por quintil do valor ajustado no modelo mpg ~ weight:")
print(media_residuo_por_quintil.round(4).to_string())
print(
       "   O padrao e em U: residuos medios positivos nas pontas e negativos no meio."
)
print(
       "   Isso antecipa nao linearidade ou erro de especificacao funcional no relacionamento entre mpg e weight."
)
print(
       "   O tratamento tipico e flexibilizar a forma funcional na Semana 7, por exemplo com termo quadratico, transformacoes ou splines, e depois reavaliar os residuos."
)


Questao 5 - respostas
2) Foi usada a separacao treino/teste com train_test_split(test_size=0.25, random_state=42) no modelo completo com as cinco variaveis numericas e dummies de origin com drop_first=True.
3) No conjunto de teste, o modelo completo obteve MAE = 2.4850, RMSE = 3.2209 e R^2 = 0.7943.
4) O baseline que sempre prevê a media do treino obteve, no teste, MAE = 5.9552, RMSE = 7.1696 e R^2 = -0.0192.
   Em relacao ao baseline, o modelo reduziu o MAE em 58.27% e o RMSE em 55.08%.
5) Repetindo o procedimento com apenas weight, no teste temos MAE = 3.3000, RMSE = 4.0890 e R^2 = 0.6685.
   Assim, as outras variaveis acrescentam ganho de 0.8150 pontos em MAE, 0.8680 em RMSE e 0.1258 em R^2 sobre o modelo so com weight.
6) O grafico de residuo x valor ajustado foi salvo em q5_residuo_vs_ajustado.png.
   O carro de maior erro absoluto no teste foi maxda rx3, com mpg observado = 18.0, previsao = 27.2378 e erro absoluto = 9.2378.
   Ele chama atencao por ser um japan de 73, muito leve